# CSV Analyzer in Jupyter

This notebook does three things:

1. **Analyzes a CSV file** and answers questions like `average salary` or `max age`
2. **Downloads a CSV from a link** and accepts **CSV only** (JSON, HTML etc. are rejected)
3. **Runs the API and a client inside the notebook**, so you can see how they talk to each other

**How to use it:** run the cells from top to bottom with `Shift + Enter`.
Cells marked **(interactive)** ask you to type something. Type your question in the box that appears, and type `quit` to move on.

You need these packages: `pandas`, `requests`, `flask`, `matplotlib`.

Every code cell has comments (lines starting with `#`) that explain what the code does.

In [ ]:
# Run this cell only if a package is missing.
# Remove the "#" at the start of the next line to install everything.
# (In Jupyter, a line that starts with "!" runs a terminal command.)

# !pip install pandas requests flask matplotlib

In [ ]:
# ---------- Imports: the tools (libraries) we use in this notebook ----------

import threading                    # lets us run servers in the background
from urllib.parse import urlparse   # splits a link into parts (scheme, domain, path)

import pandas as pd                 # reads CSV files and does the calculations
import requests                     # sends web requests (downloads data from a link)

## Step 1: Create the example data

This cell writes `employees.csv` next to the notebook, then reads it back with pandas.

In [ ]:
# ---------- Create the example CSV file ----------

# The CSV data as plain text.
# The first line holds the column names. Every other line is one row.
# Values are separated by commas.
csv_text = """name,department,age,salary,experience
Anna,Sales,30,50000,5
Bo,Sales,40,60000,15
Cy,IT,35,80000,10
Di,IT,45,90000,20
Ed,HR,28,45000,3
Fay,HR,38,55000,12
Gus,Finance,42,75000,18
Hana,Finance,31,65000,7
Ivan,IT,27,70000,3
Jo,Sales,33,52000,8
Kim,Marketing,29,48000,4
Lena,Marketing,36,58000,11
Marco,IT,50,95000,25
Nora,Finance,26,60000,2
Omar,Sales,45,68000,20
"""

# Save the text into a file called employees.csv ("w" means write).
# "with" closes the file automatically when we are done.
with open("employees.csv", "w") as file:
    file.write(csv_text)

# Read the CSV file into a pandas table (called a DataFrame).
table = pd.read_csv("employees.csv")

# Jupyter shows the value of the last line of a cell,
# so this displays the table.
table

## Step 2: The analyzer class

- `CSVError` is our own error type, used for any problem with the data or the question.
- `CSVAnalyzer` holds a table and has one method per job:

| Method | What it does |
|---|---|
| `info()` | Number of rows and the column names |
| `calculate(action, column)` | One calculation (average, sum, max, min, median) on a number column |
| `group_by(action, column, group)` | The same calculation for each group, e.g. per department |
| `answer(question)` | Reads a text question and calls the right method |

In [ ]:
# ---------- Our own error type ----------

class CSVError(Exception):
    """Used when the data or the question is not valid.
    Having our own error lets us catch only OUR problems later."""


# ---------- The analyzer: all the analysis lives in this class ----------

class CSVAnalyzer:

    # Dictionary that translates the word the user types
    # into the name of the real pandas method.
    # Example: "average" -> "mean"
    ACTIONS = {"average": "mean", "sum": "sum", "max": "max", "min": "min", "median": "median"}

    def __init__(self, table):
        # __init__ runs when we create the object: CSVAnalyzer(table)
        # We store the table inside the object so all methods can use it.
        self.table = table

    def info(self):
        # Returns the number of rows and the column names
        return {"rows": len(self.table), "columns": list(self.table.columns)}

    def _check_number_column(self, column):
        # Helper method (the "_" means "for internal use").
        # Stops the program early if the column is wrong.
        if column not in self.table.columns:
            raise CSVError(f"Column '{column}' not found. Columns: {list(self.table.columns)}")
        # We can only do maths on columns that contain numbers
        if not pd.api.types.is_numeric_dtype(self.table[column]):
            raise CSVError(f"Column '{column}' is not a number column")

    def _check_action(self, action):
        # Helper method: the action must be one of the allowed words
        if action not in self.ACTIONS:
            raise CSVError(f"Unknown action '{action}'. Use: {', '.join(self.ACTIONS)}")

    def calculate(self, action, column):
        # One calculation on one column, e.g. calculate("average", "salary")
        self._check_action(action)
        self._check_number_column(column)

        # self.ACTIONS[action] gives the pandas method name, e.g. "mean".
        # getattr(column_data, "mean") finds that method and () runs it.
        # This does the same as column_data.mean(), but works for any action
        # without a long chain of if / elif.
        result = getattr(self.table[column], self.ACTIONS[action])()

        # pandas returns a special number type. .item() turns it into a normal
        # Python number (needed so the API can send it as JSON).
        # round(..., 2) keeps two decimals.
        return round(result.item(), 2)

    def group_by(self, action, column, group):
        # The same calculation, but once for each group (e.g. each department)
        self._check_action(action)
        self._check_number_column(column)
        if group not in self.table.columns:
            raise CSVError(f"Column '{group}' not found. Columns: {list(self.table.columns)}")

        # groupby splits the table into one part per group.
        # Then we run the calculation (mean, sum, ...) on every part.
        result = getattr(self.table.groupby(group)[column], self.ACTIONS[action])()

        # Round the numbers and convert to a dictionary like {"IT": 83750.0, ...}
        return result.round(2).to_dict()

    def answer(self, question):
        # Reads a question written as text and decides what to do.

        # "Average Salary" -> ["average", "salary"]
        words = question.lower().split()

        # One-word questions
        if words == ["count"]:
            return {"answer": len(self.table)}
        if words == ["columns"]:
            return {"answer": list(self.table.columns)}

        # Two words: action + column, e.g. "average salary"
        if len(words) == 2:
            action, column = words          # unpack the list into two variables
            return {"action": action, "column": column,
                    "answer": self.calculate(action, column)}

        # Four words with "by" in the middle: "average salary by department"
        if len(words) == 4 and words[2] == "by":
            action, column, _, group = words    # "_" = a word we do not need ("by")
            return {"action": action, "column": column, "group": group,
                    "answer": self.group_by(action, column, group)}

        # Anything else: explain the correct format
        raise CSVError("Question must look like: 'average salary' or 'average salary by department'")

## Step 3: Ask questions about the file

`ask()` sends a question to the analyzer and prints the answer or the error.

In [ ]:
# ---------- Use the analyzer on the local file ----------

# Create an analyzer object that works on our table
analyzer = CSVAnalyzer(table)
print(analyzer.info())


def ask(question, analyzer=analyzer):
    # Sends one question to the analyzer and prints the result.
    # "analyzer=analyzer" is a default value, so ask("max age") uses our table,
    # but later we can pass a different analyzer.
    try:
        result = analyzer.answer(question)
        print(f"{question}  ->  {result['answer']}")
    except CSVError as error:
        # If something is wrong, show the message instead of crashing
        print(f"{question}  ->  Error: {error}")


# Try many questions. The last two are wrong on purpose, to show the error messages.
for question in ["count", "columns", "average salary", "max age", "min experience",
                 "sum salary", "median salary", "average salary by department",
                 "average name", "average bonus"]:
    ask(question)

### Ask your own questions **(interactive)**

Examples: `average salary`, `max age`, `average salary by department`. Type `quit` to stop.

In [ ]:
# ---------- Ask your own questions ----------

while True:                                  # repeat until we hit "break"
    question = input("Your question (or quit): ").strip()   # .strip() removes extra spaces
    if question.lower() == "quit":
        print("Goodbye!")
        break                                # leave the loop
    ask(question)

### A quick chart

Notebooks can draw charts, which the console version cannot.

In [ ]:
# ---------- Draw a chart ----------

import matplotlib.pyplot as plt

# Step by step:
# 1. group the rows by department
# 2. take the average salary of each group
# 3. sort from smallest to largest
# 4. draw a horizontal bar chart ("barh")
table.groupby("department")["salary"].mean().sort_values().plot(
    kind="barh", title="Average salary by department")

plt.xlabel("Average salary")   # label under the chart
plt.show()                     # display the chart

## Step 4: Download a CSV from a link (CSV only)

`CSVFetcher.fetch(url)` follows these steps:

1. Check the link starts with `http://` or `https://`
2. Send the web request with `requests.get()`
3. Check the server answered with status 200
4. **Accept CSV only**: the link ends in `.csv` or the type contains "csv", and the type is not HTML, JSON or XML
5. Check the size (maximum 5 MB) and that pandas can read it as a table

In [ ]:
# ---------- The fetcher: downloads data and makes sure it is a CSV ----------

import io                       # lets pandas read text as if it were a file

MAX_SIZE = 5 * 1024 * 1024      # maximum file size: 5 MB (in bytes)


class CSVFetcher:
    def __init__(self, timeout=10):
        # How many seconds to wait for the other server before giving up
        self.timeout = timeout

    def fetch(self, url):
        # Downloads the link and returns a pandas table.
        # Raises CSVError if anything is wrong or the data is not CSV.

        # Check 1: is the link valid?
        # For "https://site.com/data.csv":
        #   scheme = "https", netloc = "site.com", path = "/data.csv"
        parsed = urlparse(url)
        if parsed.scheme not in ("http", "https") or not parsed.netloc:
            raise CSVError("Invalid URL. It must start with http:// or https://")

        # Check 2: send the request to the other server.
        # The timeout stops us from waiting forever if the server is slow.
        try:
            response = requests.get(url, timeout=self.timeout)
        except requests.exceptions.RequestException as error:
            raise CSVError(f"Could not download the data: {error}")

        # Check 3: status code 200 means OK (404 = not found, etc.)
        if response.status_code != 200:
            raise CSVError(f"The other server answered with status {response.status_code}")

        # Check 4: accept CSV only.
        # The server tells us what it sent in the "Content-Type" header
        # (for example "text/csv" or "application/json").
        content_type = response.headers.get("Content-Type", "").lower()
        url_is_csv = parsed.path.lower().endswith(".csv")    # link ends with .csv
        type_is_csv = "csv" in content_type                   # server says it is CSV
        type_is_other = "html" in content_type or "json" in content_type or "xml" in content_type

        # Reject if the type is clearly something else, or if nothing says it is CSV
        if type_is_other or not (url_is_csv or type_is_csv):
            raise CSVError("Only CSV data is accepted (got: " + (content_type or "unknown") + ")")

        # Check 5: file size
        if len(response.content) > MAX_SIZE:
            raise CSVError("File is too big (maximum 5 MB)")

        # Check 6: can pandas really read it as a table?
        # io.StringIO makes the downloaded text behave like a file.
        try:
            remote_table = pd.read_csv(io.StringIO(response.text))
        except Exception:
            raise CSVError("The file could not be read as a CSV table")

        if remote_table.empty:
            raise CSVError("The CSV file has no data rows")

        # All checks passed: return the table
        return remote_table

### A small "other source" on your own computer

To try the fetcher without the internet, this cell starts a tiny web server **inside the notebook** (port 8000).
It shares the notebook folder, so `employees.csv` becomes available at `http://127.0.0.1:8000/employees.csv`.
It also creates `data.json` so we can test that non-CSV data is rejected.

In [ ]:
# ---------- A tiny web server that shares the files in this folder ----------

import functools
import http.server
import socketserver

# Create a JSON file. We will use it to test that non-CSV data is rejected.
with open("data.json", "w") as file:
    file.write('{"a": 1}')


class QuietHandler(http.server.SimpleHTTPRequestHandler):
    # SimpleHTTPRequestHandler is Python's built-in handler that serves files.
    # We override log_message so it does not print a line for every request.
    def log_message(self, *args):
        pass


# Lets us restart the server on the same port without waiting
socketserver.TCPServer.allow_reuse_address = True

try:
    # 127.0.0.1 means "this computer only", 8000 is the port number.
    # functools.partial pre-fills directory="." (the notebook folder).
    file_server = socketserver.ThreadingTCPServer(
        ("127.0.0.1", 8000), functools.partial(QuietHandler, directory="."))

    # serve_forever() never ends, so we run it in a background thread.
    # daemon=True means the thread stops when the notebook closes.
    threading.Thread(target=file_server.serve_forever, daemon=True).start()
    print("File server running: http://127.0.0.1:8000/employees.csv")
except OSError:
    # This happens if something already uses port 8000
    print("Port 8000 is already in use. Run the cleanup cell at the end, or restart the kernel.")

In [ ]:
# ---------- Test the fetcher ----------

fetcher = CSVFetcher()

# 1) A real CSV link: accepted.
#    The SAME CSVAnalyzer class works on the downloaded table.
remote_table = fetcher.fetch("http://127.0.0.1:8000/employees.csv")
remote_analyzer = CSVAnalyzer(remote_table)
print(remote_analyzer.info())
ask("average salary", remote_analyzer)
ask("average salary by department", remote_analyzer)

# 2) Bad links: each one is rejected with a clear message
#    (a JSON file, a file that does not exist, and a wrong kind of link)
for bad_url in ["http://127.0.0.1:8000/data.json",
                "http://127.0.0.1:8000/nofile.csv",
                "ftp://example.com/data.csv"]:
    try:
        fetcher.fetch(bad_url)
    except CSVError as error:
        print(f"{bad_url}  ->  Error: {error}")

## Step 5: Run the API inside the notebook

Now the same classes are wrapped in a Flask web API with two endpoints:

| Endpoint | What it does |
|---|---|
| `/analyze?url=<csv link>` | Rows and columns of the CSV |
| `/ask?url=<csv link>&question=<question>` | The answer to one question |

The API runs in a background thread on port 5000, so the notebook stays usable.

In [ ]:
# ---------- The web API (Flask) ----------

import logging

from flask import Flask, jsonify, request
from werkzeug.serving import make_server

# Hide the log line that Flask prints for every request (keeps the notebook clean)
logging.getLogger("werkzeug").setLevel(logging.ERROR)

# Create the web application
app = Flask("csv_api")


# A "route" connects a link path to a function.
# When someone opens /analyze?url=..., this function runs.
@app.route("/analyze")
def analyze():
    # Read the value after "?url=" in the link
    url = request.args.get("url")
    if not url:
        # 400 = "bad request": the caller made a mistake
        return jsonify(error="Missing 'url' parameter"), 400
    try:
        # download + check it is CSV -> analyze -> send the result as JSON
        return jsonify(CSVAnalyzer(fetcher.fetch(url)).info())
    except CSVError as error:
        # Any known problem becomes a JSON error message
        return jsonify(error=str(error)), 400


# Second route: /ask?url=...&question=...
# (the function is called ask_route so it does not replace our ask() function)
@app.route("/ask")
def ask_route():
    url = request.args.get("url")
    question = request.args.get("question")
    if not url or not question:
        return jsonify(error="Missing 'url' or 'question' parameter"), 400
    try:
        return jsonify(CSVAnalyzer(fetcher.fetch(url)).answer(question))
    except CSVError as error:
        return jsonify(error=str(error)), 400


# Start the API in a background thread (same idea as the file server above)
try:
    api_server = make_server("127.0.0.1", 5000, app)
    threading.Thread(target=api_server.serve_forever, daemon=True).start()
    print("API running: http://127.0.0.1:5000")
except OSError:
    print("Port 5000 is already in use. Run the cleanup cell at the end, or restart the kernel.")

## Step 6: Use the API from code (the client)

`api_ask()` sends the CSV link and the question to the API. Passing `params=` builds the link for us
and converts spaces to `%20` automatically.

In [ ]:
# ---------- The client: sends questions to our API ----------

API_URL = "http://127.0.0.1:5000"                    # where our API runs
CSV_LINK = "http://127.0.0.1:8000/employees.csv"     # the CSV we want to analyze


def api_ask(csv_url, question):
    # Sends one question to the API.
    # Returns two values: ok (True/False) and data (the JSON answer as a dictionary).
    try:
        # params=... builds the link:  /ask?url=...&question=...
        # and converts spaces and special characters for us.
        response = requests.get(f"{API_URL}/ask",
                                params={"url": csv_url, "question": question}, timeout=20)
    except requests.exceptions.ConnectionError:
        return False, {"error": "Cannot reach the API. Did you run the Step 5 cell?"}

    # status code 200 = success. response.json() turns the JSON text into a dictionary.
    return response.status_code == 200, response.json()


# Send several questions to the API and print the results
for question in ["average salary", "max age", "count", "average salary by department", "average name"]:
    ok, data = api_ask(CSV_LINK, question)
    print(f"{question}  ->  {data['answer'] if ok else 'Error: ' + data['error']}")

# The API refuses anything that is not CSV
ok, data = api_ask("http://127.0.0.1:8000/data.json", "count")
print("data.json  ->", data)

### Ask the API your own questions **(interactive)**

Type a question and the notebook sends it to the API. Type `quit` to stop.

In [ ]:
# ---------- Ask the API your own questions ----------

# If you just press Enter, an empty text counts as "false",
# so "or CSV_LINK" uses the example link instead.
csv_link = input("CSV link (press Enter to use the example): ").strip() or CSV_LINK

while True:
    question = input("Your question (or quit): ").strip()
    if question.lower() == "quit":
        print("Goodbye!")
        break
    ok, data = api_ask(csv_link, question)
    print("Answer:", data["answer"] if ok else "Error: " + data["error"])

You can also test the API in your browser while the notebook is running:

```
http://127.0.0.1:5000/analyze?url=http://127.0.0.1:8000/employees.csv
http://127.0.0.1:5000/ask?url=http://127.0.0.1:8000/employees.csv&question=average salary
```

## Cleanup

Run this cell when you are done. It stops both servers and frees ports 5000 and 8000.
(Restarting the kernel does the same.)

In [ ]:
# ---------- Stop both servers and free the ports ----------

for name in ["api_server", "file_server"]:
    # globals().get(name) looks up a variable by its name.
    # It returns None if the variable does not exist (server never started).
    server = globals().get(name)
    if server is not None:
        server.shutdown()        # stop the server loop
        server.server_close()    # release the port
        print(name, "stopped")

## What next?

- Replace `employees.csv` with your own file (use lowercase column names for easy questions)
- Add a `count by department` question
- Show more charts with `matplotlib`